# Tutorial: Privacy-Preserving Data Science with Differential Privacy

This notebook tutorial focuses on Differential Privacy. We will demonstrate how adding a tiny bit of calculated "noise" can protect an individual's data while still giving us an accurate result for the group.

### Objective: Calculate an average salary without "leaking" the salary of any specific individual.

## 1. The Scenario: The Privacy Leak

Imagine we have a small company of 10 employees. We want to publish the **average salary**. However, if a high-earner (like the CEO) joins or leaves, the average changes so drastically that everyone can figure out exactly what that person makes.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Original salaries (9 employees)
salaries = [45000, 50000, 52000, 48000, 55000, 60000, 58000, 47000, 51000]

# 2. A "Billionaire" CEO joins the dataset
salaries_with_ceo = salaries + [1_000_000]

print(f"Original Average: ${np.mean(salaries):,.2f}")
print(f"Average with CEO: ${np.mean(salaries_with_ceo):,.2f}")

By simply looking at the difference between these two averages, an outsider can mathematically back-calculate the CEO's exact salary. This is a **privacy leak**.

---

## 2. Enter Differential Privacy (DP)

Differential Privacy solves this by adding **statistical noise**. The goal is that the result of a query (like "What is the average?") should be roughly the same whether any one individual is included in the dataset or not.

### The Key Variable: Epsilon ($\epsilon$)

$\epsilon$ is your **Privacy Budget**.

* **Small $\epsilon$ (e.g., 0.1):** High privacy, more noise, less accuracy.
* **Large $\epsilon$ (e.g., 10.0):** Low privacy, less noise, high accuracy.

---

## 3. Implementing the Laplace Mechanism

To make the average "private," we add noise sampled from a **Laplace Distribution**. The amount of noise depends on the **Sensitivity** (the maximum amount one person can change the result).

In [38]:
def private_mean(data, epsilon, upper_bound):
    """
    Calculates a DP mean.
    upper_bound: The maximum possible salary (to calculate sensitivity)
    """
    actual_mean = np.mean(data)

    # Sensitivity of a mean is: (Max - Min) / N
    sensitivity = upper_bound / len(data)

    # Scale of the Laplace noise
    scale = sensitivity / epsilon

    # Add noise
    noise = np.random.laplace(0, scale)
    return actual_mean + noise

# Let's test it
epsilon = 0.8
max_salary = 1_000_000 # We assume $1M is the max possible salary

dp_result = private_mean(salaries_with_ceo, epsilon, max_salary)
print(f"Differentially Private Average: ${dp_result:,.2f}")

---

## 4. Visualizing the Accuracy vs. Privacy Trade-off

Let's see what happens to our data when we vary the privacy budget.

In [3]:
epsilons = [0.1, 0.5, 1.0, 5.0]
results = {e: [private_mean(salaries_with_ceo, e, 1_000_000) for _ in range(100)] for e in epsilons}

plt.figure(figsize=(10, 6))
for e, vals in results.items():
    plt.hist(vals, bins=30, alpha=0.5, label=f'Epsilon = {e}')

plt.axvline(np.mean(salaries_with_ceo), color='red', linestyle='--', label='True Average')
plt.title("Distribution of Private Averages (100 trials)")
plt.xlabel("Salary Value")
plt.legend()
plt.show()

### Observations:

1. **At $\epsilon=0.1$:** The "spread" is huge. The individual's data is very safe because the noise is so high, but the average is less useful.
2. **At $\epsilon=5.0$:** The distribution narrows around the true average. The data is more useful, but the "privacy shield" is thinner.